# Ocean Emulator UNet: Forward Ocean Heat Content Rollout with ACCESS-OM2

This notebook trains a clean **autoregressive forward emulator** for ocean heat
content using ACCESS-OM2 data. Given a short window of past OHC states and the
current surface forcing, the model predicts the next OHC state. Chaining this
one-step predictor forward produces a continuous OHC rollout.

Surface forcing is the total surface heat flux, with wind stress (`tau_x`,
`tau_y`) available as an optional additional forcing input (see
`INCLUDE_WIND_STRESS_FORCING` below) -- when enabled, wind stress is selected,
normalised, and channel-stacked identically to the heat flux everywhere in the
pipeline.

The model used here is deliberately simple: a deterministic `ForwardUNet` wrapper
around the existing `UNet` / `PartialConv2d` architecture from `Emulator.py`.
The training path is baseline-only.

---
## Data & normalisation

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: import dependencies and define paths/device for the full
# forward-emulator workflow. Extends the import list from AutoEncoder_om2.ipynb
# with petpipe.modifications and petpipe.branching, used below to build the
# sliding-window / state-forcing-split pipeline.
# Outcome: pipeline/data/model/plot modules are loaded and local utils are importable.
# -----------------------------------------------------------------------------
# System and path handling
import sys
import functools
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd
import xarray as xr

# PyEarthTools pipeline
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import pyearthtools.training

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
import lightning as L

# Visualization
import matplotlib.pyplot as plt
import warnings

# Configure device and random seed
torch.manual_seed(42)


def cuda_device_is_supported():
    if not torch.cuda.is_available():
        return False

    device_capability = torch.cuda.get_device_capability(0)
    device_arch = f"sm_{device_capability[0]}{device_capability[1]}"
    supported_arches = torch.cuda.get_arch_list()
    return not supported_arches or device_arch in supported_arches


cuda_is_usable = cuda_device_is_supported()
if torch.cuda.is_available() and not cuda_is_usable:
    device_capability = torch.cuda.get_device_capability(0)
    warnings.warn(
        "CUDA is visible, but this PyTorch build does not support "
        f"GPU compute capability {device_capability[0]}.{device_capability[1]}; "
        "falling back to CPU. Use an H100 or another supported GPU for CUDA training."
    )

device = torch.device("cuda:0" if cuda_is_usable else "cpu")

# Source data and code directories
database = "/g/data/nm47/txs156/"
# Taimoor's repobase:
# repobase = "/home/156/txs156/uom/OM2-emulator/"
# Ryan's repobase:
repobase = "/home/561/rmh561/ML/OM2-emulator/"
database = "/g/data/dx2/rmh561/"
# Navid's repobase:
# repobase = "/home/552/nc3020/gdata/OM2-emulator/"

# Add the src directory to the Python path (relative to notebook location)
src_path = Path(repobase) / "src"
sys.path.insert(0, str(src_path))

# Import local OM2 emulator modules
# Same building blocks as AutoEncoder_om2.ipynb -- the UNet and PartialConv2d
# architecture are kept as-is. AutoEncoder is no longer used: the forward emulator
# is not a reconstruction model, so there is no single-frame autoencoding step.
from Data import ACCESS_OHC, build_normalisation, make_fast_dl
from Emulator import LightningWrapper, PartialConv2d, UNet


Unchanged in approach from `AutoEncoder_om2.ipynb`: the same `build_normalisation`
strategy is applied whether the model reconstructs one frame or predicts the next
one. The one addition here is `forcing_variables`, set below, which lists which
surface fields are treated as forcing -- total surface heat flux always, with
`tau_x`/`tau_y` appended when `INCLUDE_WIND_STRESS_FORCING` is True. Every variable
in that list is normalised the same way and channel-stacked in the forcing branch.

In [ ]:
# -----------------------------------------------------------------------------
# Notebook-wide time configuration.
#
# Every date range used anywhere below -- pipeline construction, the train/
# valid/skill-test/control splits, and the final rollout diagnostics -- is
# derived from the boundaries set here, so changing the training period only
# requires editing this cell.
#
#   time_start   -- first month of usable data.
#   val_start    -- first month of the fully held-out skill-test window.
#   VALID_MONTHS -- length (in months) of the validation-during-training
#                   window, taken immediately before val_start. train_end
#                   (below) is derived from this, not set directly, so the
#                   validation window can never accidentally shrink to
#                   near-nothing just because train_end/val_start were picked
#                   close together.
#   time_end     -- last month of usable data.
# -----------------------------------------------------------------------------
time_start = '1970-01'
val_start = "2005-02"
VALID_MONTHS = 12
time_end = '2018-12'
time_interval = '1MS'  # pandas frequency string, used only by build_normalisation's pd.date_range

# PET's DateRange/TimeDelta/TemporalRetrieval spell "monthly" differently to
# pandas's '1MS' above -- set once here rather than repeating the literal
# "month" string in every pipeline/iterator cell below.
ROLLOUT_INTERVAL = "1 month"
ROLLOUT_DELTA_UNIT = "month"


def _shift_months(yyyymm, months):
    """Shift a 'YYYY-MM' string by a whole number of months."""
    return (pd.Timestamp(yyyymm) + pd.DateOffset(months=months)).strftime("%Y-%m")


# state_window uses prior_indexes=[-1, 0], so the first queryable sample needs
# one prior month of context -- every pipeline that walks forward from
# time_start therefore actually starts one month later.
PIPELINE_START = _shift_months(time_start, 1)

# train_end is the last month used both for training samples and for
# computing normalisation mean/std (the same boundary avoids data leakage
# into the training normalisation stats). It is derived from val_start and
# VALID_MONTHS -- not set independently -- so training and validation can
# never silently overlap or collapse to a 1-month validation window.
train_end = _shift_months(val_start, -VALID_MONTHS)

# Training pipeline samples: PIPELINE_START through train_end.
TRAIN_SPLIT_START = PIPELINE_START
TRAIN_SPLIT_END = train_end

# Validation-during-training samples: the VALID_MONTHS immediately before
# val_start.
VALID_SPLIT_START = train_end
VALID_SPLIT_END = val_start

# Skill-test / full rollout: real, held-out forcing only -- this is also the
# window the full autoregressive rollout (further below) is evaluated over,
# so it never re-walks the training or validation months. It is seeded at
# val_start (the last real ground-truth state before the held-out window)
# and its targets run from one month after val_start through time_end.
SKILL_TEST_START = val_start
SKILL_TEST_TARGET_START = _shift_months(val_start, 1)
SKILL_TEST_TARGET_END = time_end

# Control-rollout window: a single fixed calendar year chosen for near-zero
# net surface heat flux (RYF0304 from Stewart et al., 2020) -- independent of
# the training period above, so it is not derived from time_start/time_end.
CONTROL_WINDOW_START = "2003-01"
CONTROL_WINDOW_END = "2004-01"


In [ ]:
# Wind stress (tau_x, tau_y) forcing is optional. Set to False to reproduce the
# original heat-flux-only forward emulator. When True, tau_x and tau_y are
# treated as additional forcing variables -- selected, normalised, and
# channel-stacked identically to total_surface_heat_flx everywhere below.
INCLUDE_WIND_STRESS_FORCING = True
forcing_variables = ["total_surface_heat_flx"] + (
    ["tau_x", "tau_y"] if INCLUDE_WIND_STRESS_FORCING else []
)
# total_surface_heat_flx is always first in forcing_variables, so this is the
# channel index the global heat-closure penalty reads forcing from below.
HEAT_FLUX_CHANNEL_INDEX = forcing_variables.index("total_surface_heat_flx")

norm_variables = ["ocean_heat_content_2d", *forcing_variables]
norm_strat = "Spatial_climatology"
datapath = database + "OM2-emulator/data/1deg_ocean_heat_emulator_data.nc"

mask, normalisation = build_normalisation(datapath, norm_strat, norm_variables, time_window = dict(start=time_start, end=time_end, freq=time_interval),\
                                          train_end = train_end, mask = True)
mean = normalisation._initialisation["mean"]
deviation = normalisation._initialisation["deviation"]

In [ ]:
# Set up accessor for the ACCESS data
ACCESS_OHC_accessor = ACCESS_OHC(
    ["area_t", "ocean_heat_content_2d", *forcing_variables],
    root=database + "OM2-emulator/data/",
)


## Setup the forward-emulation PET pipeline

This is the core structural change from `AutoEncoder_om2.ipynb`. The original pipeline
iterates over single dates and emits one `(OHC, flux)` frame per sample, which is fine
for reconstruction but has no notion of "predict the next state given recent history and
forcing."

A hand-rolled windowing wrapper around `DateRange` was the first instinct here, but PET
already has purpose-built, tested classes for exactly this pattern, in
`pyearthtools.pipeline.modifications` (`petpipe.modifications`):

- **`TemporalWindow`** — built specifically, per its own docstring, "to provide the
  ability to perform sequence-to-sequence modelling from a data accessor or pipeline
  that was designed to produce single time steps." Given `prior_indexes` and
  `posterior_indexes` (each multiplied by a `timedelta` and applied to the queried
  reference date), it returns a `(prior, posterior)` tuple directly. This is the natural
  fit for "two past states in, one future state out," matching the
  `Φ̃_{t+(n-1)Δt}, Φ̃_{t+nΔt} → Φ̃_{t+(n+1)Δt}` recurrence in Samudra (their Eq. 1).
- **`SequenceRetrieval` / `TemporalRetrieval`** — the more general, lower-level
  mechanism `TemporalWindow` is built on. Useful if a non-contiguous or asymmetric
  sampling pattern is ever needed, but `TemporalWindow`'s narrower interface is the
  better fit for this fixed two-in-one-out window and is used below.

For separating OHC (a *state* to be windowed and predicted) from surface flux (a
*forcing* that is supplied, never reconstructed), PET's `petpipe.branching` module
provides `PipelineBranchPoint`: it forks the same upstream sample down independent
sub-pipelines and returns their results as a tuple, in declared order. This replaces
what would otherwise be a manual "predict everything, then mask flux out of the loss"
workaround with two clean, independently-composed branches:

1. a **state branch**: `SelectDataset(["ocean_heat_content_2d"])` &rarr; `TemporalWindow`
2. a **forcing branch**: `SelectDataset(["total_surface_heat_flx"])`, supplying only the
   forcing at the timesteps the rollout actually needs

Each branch is itself an ordinary PET pipeline, so the existing transforms, the
normalisation step, `FillNan`, and `ToNumpy` conversion are reused unchanged within each
branch — nothing about those steps needs to be rewritten, only re-arranged around the
new branch structure.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: define the two re-usable sub-pipeline step sequences (state, forcing)
# that will be forked from the same upstream accessor inside a PipelineBranchPoint.
# Outcome: `state_steps` and `forcing_steps` are tuples of PET pipeline steps, each
# independently valid as the body of a Pipeline(...) call.
# -----------------------------------------------------------------------------

# Both branches start from the same coordinate clean-up as AutoEncoder_om2.ipynb,
# then diverge at the variable-selection step. xu_ocean/yu_ocean are dropped
# alongside geolat_t/geolon_t: tau_x/tau_y were regridded from the u-cell grid
# onto the t-cell grid in Extract_om2_data.ipynb, but xarray's `.interp` leaves
# the old xu_ocean/yu_ocean coordinate names behind as vestigial non-dimension
# coordinates riding along the new xt_ocean/yt_ocean dims -- harmless in xarray,
# but ToNumpy() chokes on a coordinate whose size doesn't match any dimension
# of a data-var-only selection (e.g. the OHC-only state branch).
_shared_coord_drop = petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t', 'xu_ocean', 'yu_ocean'], ignore_missing=True)
_shared_var_drop = petdata.transforms.variables.Drop(['area_t'])

# State branch: OHC only. This is the field that is windowed in time (two past
# states as input context, matching Samudra's 2-input/2-output recurrence) and is
# the model's prediction target.
state_steps = (
    _shared_coord_drop,
    _shared_var_drop,
    petpipe.operations.xarray.select.SelectDataset(["ocean_heat_content_2d"]),
    petpipe.operations.xarray.Sort(order=["ocean_heat_content_2d"], strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t', 'xu_ocean', 'yu_ocean'], ignore_missing=True),
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
)

# Forcing branch: known boundary conditions the emulator conditions on -- per
# Subel & Zanna (2024), surface heat flux and wind stress are what prevent an
# emulator from drifting in the absence of any external driving signal. Every
# variable in `forcing_variables` (set above -- just total_surface_heat_flx, or
# with tau_x/tau_y appended when INCLUDE_WIND_STRESS_FORCING is True) is
# selected, sorted into that fixed order, and normalised the same way, so
# ToNumpy() channel-stacks them identically regardless of how many are active.
# None of them are ever reconstructed, so this branch is not windowed across
# (t-dt, t) the way the state branch is -- only the forcing at the *target*
# timestep is needed to predict that timestep's OHC.
forcing_steps = (
    _shared_coord_drop,
    _shared_var_drop,
    petpipe.operations.xarray.select.SelectDataset(forcing_variables),
    petpipe.operations.xarray.Sort(order=forcing_variables, strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t', 'xu_ocean', 'yu_ocean'], ignore_missing=True),
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
)


In [ ]:
N_ROLLOUT_STEPS = 12

rollout_timedelta = petdata.time.TimeDelta((1, ROLLOUT_DELTA_UNIT))

# -------------------------
# STATE PIPELINE
# -------------------------

state_window = petpipe.modifications.TemporalWindow(
    prior_indexes=[-1, 0],
    posterior_indexes=list(range(1, N_ROLLOUT_STEPS + 1)),
    timedelta=rollout_timedelta,
    merge_method=functools.partial(np.concatenate, axis=0),
)

state_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *state_steps,
    state_window,
)


# -------------------------
# FORCING PIPELINE
# -------------------------

forcing_retrieval = petpipe.modifications.TemporalRetrieval(
    (1, N_ROLLOUT_STEPS),
    delta_unit=ROLLOUT_DELTA_UNIT,
    concat=True,
)

forcing_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *forcing_steps,
    forcing_retrieval,
)

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: assemble the full forward-emulation pipeline. Each branch is a
# complete source pipeline so index-aware steps such as TemporalWindow receive
# the queried date instead of an already-retrieved sample.
# Outcome: `pipeline_i[date]` returns a nested tuple
#   ( (prior_states, posterior_states), forcing_at_t )
# where prior_states has shape (2, 1, H, W), posterior_states has shape
# (1, 1, H, W), and forcing_at_t has shape (1, 1, H, W).
# -----------------------------------------------------------------------------

#This pipeline ensures we can do 'undo' at the end for the held out times
skill_target_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *state_steps,
    iterator=petpipe.iterators.DateRange(SKILL_TEST_TARGET_START, SKILL_TEST_TARGET_END, interval=ROLLOUT_INTERVAL))

pipeline_i = petpipe.Pipeline(
    petpipe.branching.PipelineBranchPoint(
        state_pipeline,    # branch 0: windowed OHC state
        forcing_pipeline,  # branch 1: flux forcing, single timestep
    ),
    # PIPELINE_START is already one month after time_start (see the time
    # configuration cell above), because state_window uses prior_indexes=[-1, 0].
    iterator=petpipe.iterators.DateRange(PIPELINE_START, time_end, interval=ROLLOUT_INTERVAL),
)


## Define the training, skill-test, and control-rollout splits

Samudra's evaluation protocol uses two separate long rollouts, deliberately kept apart
because they isolate different failure modes:

1. An **8-year skill-test rollout** against real, held-out atmospheric/surface forcing,
   used to measure how well the emulator tracks genuine forced trends.
2. A **century-scale control rollout** against *repeated* forcing from a window chosen
   specifically for near-zero net heat flux. This isolates pure equilibrium/numerical
   drift from drift caused by a genuinely changing forcing signal.

Conflating these into a single long rollout against real forcing would make it
impossible to tell whether divergence from ground truth reflects the model failing to
track a real trend, or the model drifting under flat forcing -- so both are defined
explicitly below, using PET's native `DateRange` and `SuperIterator`.

The training split itself is unchanged in spirit from `AutoEncoder_om2.ipynb` -- same
PET `DateRange` iterator class, same general boundary dates -- only the validation
window is trimmed slightly to leave room for the two held-out rollout regimes.


In [ ]:
# We define the training / validation / skill-test / control-rollout splits here.
# All boundaries are set once in the time-configuration cell above (right after
# the imports) -- see that cell's comments for what each one means.
#
# `train_split`   -- used for gradient updates.
# `valid_split`   -- used for early stopping / monitoring during training.
# `full_rollout_split` -- drives the full autoregressive rollout further below,
#                          seeded at val_start and walking only the held-out
#                          skill-test window (never training or validation
#                          months) for drift diagnostics.
# `control_window`   -- a single near-zero-net-flux window, chosen the same way as
#                        Samudra's control run, to be REPEATED via SuperIterator below
#                        rather than iterated once.

splits = {
    "train_split": petpipe.iterators.DateRange(
        TRAIN_SPLIT_START, TRAIN_SPLIT_END, interval=ROLLOUT_INTERVAL
    ),
    "valid_split": petpipe.iterators.DateRange(
        VALID_SPLIT_START, VALID_SPLIT_END, interval=ROLLOUT_INTERVAL
    ),
}

full_rollout_split = petpipe.iterators.DateRange(
    SKILL_TEST_START, _shift_months(time_end, -1), interval=ROLLOUT_INTERVAL
)


# Control-rollout window: a single fixed year chosen for near-zero net surface
# heat flux, the same rationale as Samudra Sec 2.5. The actual near-zero-flux
# window for this dataset should be identified empirically from
# `total_surface_heat_flx` (e.g. by inspecting its area-weighted annual mean
# over the training period) -- CONTROL_WINDOW_START/END (set above) are a
# placeholder to be confirmed against the real data before the control
# rollout is run.
control_window = petpipe.iterators.DateRange(
    CONTROL_WINDOW_START, CONTROL_WINDOW_END, interval=ROLLOUT_INTERVAL
) # We use RYF0304 from Stewart et al., 2020

# Repeat the control window N times via SuperIterator, chaining the same
# DateRange back-to-back -- this is the PET-native equivalent of Samudra's
# "repeat 10-year cycle" control protocol, built from the existing DateRange
# class rather than a bespoke repeating-iterator wrapper.
n_control_repeats = 100  # 100 repeats of a 1-year window = a century-scale rollout,
                         # matching the order of magnitude of Samudra's century run.
control_iterator = petpipe.iterators.SuperIterator(
    *([control_window] * n_control_repeats)
)


## Define the forward-emulator architecture

The base `UNet` and `PartialConv2d` land-handling from `Emulator.py` are kept as-is.
This notebook only wraps the UNet with the forward-emulation channel contract:
two prior OHC states plus one forcing channel in, one future OHC state out.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: thin wrapper around the existing UNet, rewiring channel counts for
# the forward-emulation contract (2 past states + forcing in, 1 future state out)
# and handling the reshape from the pipeline's nested output structure.
# -----------------------------------------------------------------------------

class ForwardUNet(nn.Module):
    """
    Forward one-step OHC emulator.

    Wraps the existing UNet from Emulator.py and optionally refines the latent
    bottleneck before decoding; only
    the input/output channel counts differ from the reconstruction model in
    AutoEncoder_om2.ipynb. PartialConv2d land-handling is inherited from the base
    UNet implementation and is not modified here.

    forward() expects:
        prior_states : (B, 2, H, W)                    -- OHC(t-dt), OHC(t), channel-stacked
        forcing      : (B, n_forcing_channels, H, W)   -- forcing(t), e.g. flux(t)
                                                           alone, or flux(t) with
                                                           tau_x(t)/tau_y(t) channel-stacked
        mask         : (H, W), (B, H, W), or (B, 1, H, W) ocean mask
    and returns:
        next_state   : (B, 1, H, W)  -- predicted OHC(t+dt)
    """

    def __init__(
        self,
        n_prior_states: int = 2,
        n_forcing_channels: int = 1,
        latent_processor: nn.Module | None = None,
    ):
        super().__init__()
        self.n_prior_states = n_prior_states
        self.n_forcing_channels = n_forcing_channels
        self.base_unet = UNet(
            input_channel_count=n_prior_states + n_forcing_channels,
            output_channel_count=1,
            latent_processor=latent_processor,
        )

    def forward(self, prior_states: torch.Tensor, forcing: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = torch.cat([prior_states, forcing], dim=1)
        mask = mask.to(device=x.device, dtype=x.dtype)
        if mask.ndim == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)
        elif mask.ndim == 3:
            mask = mask.unsqueeze(1)
        mask = mask.expand(x.shape[0], 1, x.shape[2], x.shape[3])
        return self.base_unet(x, mask)

## Multi-step rollout loss

Single-step masked MSE, as used in `AutoEncoder_om2.ipynb`, is replaced with a
multi-step recurrent rollout loss. The model's own predictions are fed back into
the next step, so compounding autoregressive error is visible during training.


In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: multi-step recurrent rollout loss for the deterministic ForwardUNet.
# Replaces the single-step masked MSE from AutoEncoder_om2.
# -----------------------------------------------------------------------------

def expand_ocean_mask(mask, like):
    valid_mask = mask.to(device=like.device, dtype=like.dtype)
    if valid_mask.ndim == 2:
        valid_mask = valid_mask.unsqueeze(0).unsqueeze(0)
    elif valid_mask.ndim == 3:
        valid_mask = valid_mask.unsqueeze(1)
    return valid_mask.expand_as(like)


def squeeze_field_axes(field):
    """Drop singleton variable/channel axes, preserving batch and spatial axes."""
    while field.ndim > 3:
        squeezed = False
        for dim in range(1, field.ndim - 2):
            if field.shape[dim] == 1:
                field = field.squeeze(dim)
                squeezed = True
                break
        if not squeezed:
            break
    return field


def physical_field(normalised, mean_lookup, std_lookup, time_index):
    normalised = squeeze_field_axes(normalised)
    mean_t = mean_lookup[time_index].to(device=normalised.device, dtype=normalised.dtype)
    std_t = std_lookup[time_index].to(device=normalised.device, dtype=normalised.dtype)
    return normalised * std_t + mean_t


def global_cumulative_closure_loss(
    initial_ohc_norm,
    predicted_ohc_norm,
    forcing_sequence_norm,
    initial_time_index,
    target_time_index,
    forcing_time_indices,
    area,
    mask,
    ohc_mean,
    ohc_std,
    forcing_mean,
    forcing_std,
    dt_seconds,
    surface_flux_sign=1.0,
    closure_min_scale=1.0e20,
    heat_flux_channel_index=0,
):
    """
    Area-integrated cumulative heat-budget closure over a rollout segment.

    This compares OHC[target] - OHC[initial] against the time integral of the
    intervening surface heat fluxes. It deliberately does not enforce one-month
    pointwise closure, because the monthly fields are not an exact discrete budget
    at that cadence.

    `forcing_sequence_norm` may carry more than one forcing channel (e.g. heat
    flux stacked with wind stress); `forcing_mean`/`forcing_std` are always the
    heat-flux variable's own normalisation statistics, so only the heat-flux
    channel (`heat_flux_channel_index`) is read out of it here -- wind stress
    does not enter a heat-budget closure term.
    """
    area = area.to(device=predicted_ohc_norm.device, dtype=predicted_ohc_norm.dtype)
    ocean_mask = mask.to(device=predicted_ohc_norm.device, dtype=predicted_ohc_norm.dtype)
    area = area * ocean_mask

    initial_ohc = physical_field(
        initial_ohc_norm, ohc_mean, ohc_std, initial_time_index
    )
    predicted_ohc = physical_field(
        predicted_ohc_norm, ohc_mean, ohc_std, target_time_index
    )
    ohc_change_global = ((predicted_ohc - initial_ohc) * area).sum(dim=(-2, -1))

    forcing_integral_global = 0.0
    for step in range(forcing_sequence_norm.shape[1]):
        # forcing_sequence_norm is (B, steps, C, H, W); select the heat-flux
        # channel before unnormalising -- forcing_mean/forcing_std only cover
        # that one variable.
        heat_flux_norm = forcing_sequence_norm[:, step, heat_flux_channel_index]
        forcing_t = physical_field(
            heat_flux_norm,
            forcing_mean,
            forcing_std,
            forcing_time_indices[:, step],
        )
        forcing_integral_global = forcing_integral_global + (
            (forcing_t * area).sum(dim=(-2, -1)) * dt_seconds * surface_flux_sign
        )

    residual = ohc_change_global - forcing_integral_global

    # The residual is in Joules. Scale by the larger of accumulated forcing or
    # realised OHC change so weak-forcing windows do not explode the regulariser.
    scale = torch.maximum(
        forcing_integral_global.detach().abs().mean(),
        ohc_change_global.detach().abs().mean(),
    ).clamp_min(closure_min_scale)
    return ((residual / scale) ** 2).mean()


# Backwards-compatible name for old cells; now uses cumulative rollout closure.
global_closure_loss = global_cumulative_closure_loss


def rollout_loss(
    model,
    initial_prior_states,
    forcing_sequence,
    target_sequence,
    mask,
    n_steps=4,
    target_time_indices=None,
    area=None,
    ohc_mean=None,
    ohc_std=None,
    forcing_mean=None,
    forcing_std=None,
    dt_seconds=30 * 24 * 60 * 60,
    closure_weight=0.0,
    surface_flux_sign=1.0,
    closure_min_scale=1.0e20,
    heat_flux_channel_index=0,
):
    """
    Multi-step recurrent rollout loss.

    The model's own predictions are fed back as input for n_steps passes. The
    primary term is masked normalised MSE. Optionally, a global cumulative heat
    closure penalty discourages loss of the integrated forced signal over the
    rollout window.

    `forcing_sequence` is (B, n_steps, n_forcing_channels, H, W) -- one or more
    channel-stacked forcing variables (e.g. heat flux alone, or heat flux with
    wind stress). All channels are fed to the model at every step; only the
    heat-flux channel (`heat_flux_channel_index`) is used by the closure term.
    """
    prior_states = initial_prior_states
    initial_ohc_norm = initial_prior_states[:, 1:2]
    mse_total = 0.0
    closure_total = 0.0
    use_closure = closure_weight > 0

    if use_closure and target_time_indices is None:
        raise ValueError(
            "closure_weight > 0 requires target_time_indices in each batch. "
            "Rebuild or reload the rollout cache with target_time_indices."
        )

    if use_closure:
        initial_time_index = target_time_indices[:, 0].to(initial_prior_states.device) - 1

    for step in range(n_steps):
        forcing_t = forcing_sequence[:, step]
        target_t = target_sequence[:, step : step + 1]

        pred_t = model(prior_states, forcing_t, mask)
        step_err = (pred_t - target_t) ** 2
        valid_mask = expand_ocean_mask(mask, step_err)
        step_mse = (step_err * valid_mask).sum() / valid_mask.sum().clamp_min(1.0)
        mse_total = mse_total + step_mse

        if use_closure:
            target_idx = target_time_indices[:, step].to(pred_t.device)
            closure_total = closure_total + global_cumulative_closure_loss(
                initial_ohc_norm=initial_ohc_norm,
                predicted_ohc_norm=pred_t,
                forcing_sequence_norm=forcing_sequence[:, : step + 1],
                initial_time_index=initial_time_index,
                target_time_index=target_idx,
                forcing_time_indices=target_time_indices[:, : step + 1].to(pred_t.device),
                area=area,
                mask=mask,
                ohc_mean=ohc_mean,
                ohc_std=ohc_std,
                forcing_mean=forcing_mean,
                forcing_std=forcing_std,
                dt_seconds=dt_seconds,
                surface_flux_sign=surface_flux_sign,
                closure_min_scale=closure_min_scale,
                heat_flux_channel_index=heat_flux_channel_index,
            )

        # Feed prediction back in as context for the next step.
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

    mse_loss = mse_total / n_steps
    closure_loss = closure_total / n_steps
    return mse_loss + closure_weight * closure_loss


## Lightning wrapper for autoregressive training

`LightningWrapper` from `Emulator.py` was built for single-frame reconstruction and is
not reused directly here. `ForwardLightningWrapper` below carries over the same
Lightning conventions (constructor signature, Adam optimiser, masked loss) and adapts
the training step to unpack the nested pipeline output and call `rollout_loss`.

In [ ]:
def as_rollout_tensor(x):
    """Convert PET/PyTorch rollout batch variants to tensors."""
    if torch.is_tensor(x):
        return x

    if isinstance(x, np.ndarray):
        return torch.as_tensor(x)

    if isinstance(x, (tuple, list)):
        if len(x) == 1:
            return as_rollout_tensor(x[0])

        xs = [as_rollout_tensor(xi) for xi in x]
        if all(torch.is_tensor(xi) for xi in xs):
            shapes = [tuple(xi.shape) for xi in xs]
            if all(shape == shapes[0] for shape in shapes):
                return torch.stack(xs, dim=1)
            raise ValueError(
                "Cannot convert a mixed-shape sequence to one rollout tensor: "
                f"got shapes {shapes}. This is probably a (prior, posterior) "
                "state pair and should be unpacked before tensor conversion."
            )

    return torch.as_tensor(x)


def is_state_pair(x):
    return isinstance(x, (tuple, list)) and len(x) == 2


def first_spatial_tensor(*items):
    """Return the first tensor-like item with batch/time/spatial dimensions."""
    for item in items:
        try:
            tensor = as_rollout_tensor(item)
        except (TypeError, ValueError):
            continue
        if torch.is_tensor(tensor) and tensor.ndim >= 4:
            return tensor

    shapes = []
    for item in items:
        try:
            tensor = as_rollout_tensor(item)
            shapes.append(tuple(tensor.shape) if torch.is_tensor(tensor) else type(tensor).__name__)
        except Exception as err:
            shapes.append(f"{type(item).__name__}: {err}")
    raise ValueError(f"Could not identify forcing tensor from candidates: {shapes}")


def unpack_rollout_batch(batch):
    """Return prior, posterior, forcing, and optional target time indices."""
    if not isinstance(batch, (tuple, list)):
        raise TypeError(f"Expected rollout batch to be tuple/list, got {type(batch)!r}")

    target_time_indices = None
    if len(batch) == 3 and is_state_pair(batch[0]):
        state_batch, forcing, target_time_indices = batch
        prior_states, posterior_states = state_batch
        return (
            as_rollout_tensor(prior_states),
            as_rollout_tensor(posterior_states),
            as_rollout_tensor(forcing),
            as_rollout_tensor(target_time_indices).long(),
        )

    if len(batch) == 2:
        state_batch, forcing = batch
        if not is_state_pair(state_batch):
            raise ValueError(
                "Expected nested rollout batch as ((prior, posterior), forcing); "
                f"got state component of type {type(state_batch)!r}"
            )
        prior_states, posterior_states = state_batch
        return (
            as_rollout_tensor(prior_states),
            as_rollout_tensor(posterior_states),
            as_rollout_tensor(forcing),
            target_time_indices,
        )

    if len(batch) == 3:
        prior_states, posterior_states, forcing = batch
        return (
            as_rollout_tensor(prior_states),
            as_rollout_tensor(posterior_states),
            as_rollout_tensor(forcing),
            target_time_indices,
        )

    raise ValueError(f"Expected rollout batch with 2 or 3 top-level entries, got {len(batch)}")


def tensor_buffer(x, dtype=torch.float32):
    return torch.nan_to_num(torch.as_tensor(x, dtype=dtype), nan=0.0, posinf=0.0, neginf=0.0)


class ForwardLightningWrapper(L.LightningModule):
    """
    Lightning wrapper for the deterministic autoregressive forward emulator.

    Expects each batch as ((prior_states, posterior_states), forcing) and, when
    closure_weight > 0, target_time_indices for physical unnormalisation.
    """

    def __init__(
        self,
        model,
        mask,
        lr=1e-4,
        n_steps=1,
        area=None,
        ohc_mean=None,
        ohc_std=None,
        forcing_mean=None,
        forcing_std=None,
        dt_seconds=30 * 24 * 60 * 60,
        closure_weight=0.0,
        surface_flux_sign=1.0,
        closure_min_scale=1.0e20,
        heat_flux_channel_index=0,
    ):
        super().__init__()
        self.model = model
        self.register_buffer("mask", tensor_buffer(mask))
        self.lr = lr
        self.n_steps = n_steps
        self.dt_seconds = dt_seconds
        self.closure_weight = closure_weight
        self.surface_flux_sign = surface_flux_sign
        self.closure_min_scale = closure_min_scale
        # Index of the heat-flux variable within the (possibly multi-channel,
        # e.g. heat flux + wind stress) forcing tensor -- used by the closure
        # term below, which is only physically meaningful for heat flux.
        self.heat_flux_channel_index = heat_flux_channel_index

        if closure_weight > 0:
            for name, value in {
                "area": area,
                "ohc_mean": ohc_mean,
                "ohc_std": ohc_std,
                "forcing_mean": forcing_mean,
                "forcing_std": forcing_std,
            }.items():
                if value is None:
                    raise ValueError(f"closure_weight > 0 requires {name}")

        self.register_buffer("area", tensor_buffer(area if area is not None else 0.0))
        self.register_buffer("ohc_mean", tensor_buffer(ohc_mean if ohc_mean is not None else 0.0))
        self.register_buffer("ohc_std", tensor_buffer(ohc_std if ohc_std is not None else 1.0))
        self.register_buffer("forcing_mean", tensor_buffer(forcing_mean if forcing_mean is not None else 0.0))
        self.register_buffer("forcing_std", tensor_buffer(forcing_std if forcing_std is not None else 1.0))

    @staticmethod
    def _squeeze_variable_axis(x):
        return x.squeeze(2) if x.ndim == 5 and x.shape[2] == 1 else x

    def _step(self, batch):
        prior_states, posterior_states, forcing, target_time_indices = unpack_rollout_batch(batch)
        prior_states = self._squeeze_variable_axis(prior_states)
        posterior_states = self._squeeze_variable_axis(posterior_states)
        # Unlike prior/posterior states (always one variable, OHC), forcing may
        # carry more than one channel (heat flux, optionally with wind stress),
        # so its channel axis is kept rather than squeezed away: forcing stays
        # (B, n_steps, n_forcing_channels, H, W) all the way into rollout_loss.
        return rollout_loss(
            model=self.model,
            initial_prior_states=prior_states,
            forcing_sequence=forcing,
            target_sequence=posterior_states,
            mask=self.mask,
            n_steps=self.n_steps,
            target_time_indices=target_time_indices,
            area=self.area,
            ohc_mean=self.ohc_mean,
            ohc_std=self.ohc_std,
            forcing_mean=self.forcing_mean,
            forcing_std=self.forcing_std,
            dt_seconds=self.dt_seconds,
            closure_weight=self.closure_weight,
            surface_flux_sign=self.surface_flux_sign,
            closure_min_scale=self.closure_min_scale,
            heat_flux_channel_index=self.heat_flux_channel_index,
        )

    def training_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.lr)


In [ ]:
base_model = ForwardUNet(
    n_prior_states=2,
    n_forcing_channels=len(forcing_variables),
    latent_processor=None,
)

closure_area = ACCESS_OHC_accessor[time_start]["area_t"].values

# Global heat-closure penalty. The physically meaningful constraint is
# cumulative over the rollout window:
# sum((OHC[t+n] - OHC[t]) * area) ~= sum_k sum(surface_flux[t+k] * area) * dt.
# If the dataset's positive surface-flux convention is upward/out of ocean, set
# surface_flux_sign=-1.0. The known-closure diagnostic below should be inspected
# before treating this as more than a soft regulariser.
#
# forcing_mean/forcing_std below are always the heat-flux variable's own
# normalisation statistics, regardless of how many forcing channels are active
# -- the closure term reads only the heat-flux channel out of the forcing
# tensor (see HEAT_FLUX_CHANNEL_INDEX, set above alongside forcing_variables).
CLOSURE_WEIGHT = 0.1
SECONDS_PER_MONTH = 30 * 24 * 60 * 60

lightning_model = ForwardLightningWrapper(
    model=base_model,
    mask=mask.values,
    lr=1e-4,
    n_steps=N_ROLLOUT_STEPS,
    area=closure_area,
    ohc_mean=mean["ocean_heat_content_2d"].values,
    ohc_std=deviation["ocean_heat_content_2d"].values,
    forcing_mean=mean["total_surface_heat_flx"].values,
    forcing_std=deviation["total_surface_heat_flx"].values,
    dt_seconds=SECONDS_PER_MONTH,
    closure_weight=CLOSURE_WEIGHT,
    heat_flux_channel_index=HEAT_FLUX_CHANNEL_INDEX,
)


## Fast rollout dataloaders

The PET Lightning datamodule's default collation flattens this branched pipeline in
a way that separates the forcing branch from the state pair. For this forward
emulator, cache samples directly from `pipeline_i[date]` over each split so the
known branch contract `((prior_states, posterior_states), forcing)` is preserved.


In [ ]:
normalisation_time_lookup = {
    pd.Timestamp(t).strftime("%Y-%m"): i
    for i, t in enumerate(mean.time.values)
}


def rollout_target_time_indices(date, n_steps):
    base_date = pd.Timestamp(str(date))
    return torch.tensor(
        [
            normalisation_time_lookup[
                (base_date + pd.DateOffset(months=step)).strftime("%Y-%m")
            ]
            for step in range(1, n_steps + 1)
        ],
        dtype=torch.long,
    )


def rollout_target_time_index_tensor(iterator, n_steps):
    return torch.stack(
        [rollout_target_time_indices(date, n_steps) for date in iterator],
        dim=0,
    )


class RolloutTensorDataset(Dataset):
    def __init__(self, prior_states, posterior_states, forcing, target_time_indices=None):
        self.prior_states = prior_states
        self.posterior_states = posterior_states
        self.forcing = forcing
        self.target_time_indices = target_time_indices

    def __len__(self):
        return self.prior_states.shape[0]

    def __getitem__(self, idx):
        sample = (
            (self.prior_states[idx], self.posterior_states[idx]),
            self.forcing[idx],
        )
        if self.target_time_indices is None:
            return sample
        return (*sample, self.target_time_indices[idx])


def ensure_batch_axis(x):
    x = as_rollout_tensor(x)
    return x.unsqueeze(0) if x.ndim == 4 else x


from tqdm.auto import tqdm


def make_fast_rollout_dl(
    pipeline,
    iterator,
    batch_size=8,
    shuffle=False,
    drop_last=False,
    desc="Building rollout cache",
):
    prior_batches = []
    posterior_batches = []
    forcing_batches = []
    target_time_index_batches = []

    # Each date triggers its own multi-month pipeline fetch with no caching
    # across dates, so this loop can take a while with a long split -- list()
    # the iterator up front so tqdm knows the total and can show a real ETA.
    dates = list(iterator)
    for date in tqdm(dates, desc=desc):
        prior_states, posterior_states, forcing, _ = unpack_rollout_batch(
            pipeline[date]
        )

        prior_batches.append(
            ensure_batch_axis(prior_states).detach().cpu()
        )
        posterior_batches.append(
            ensure_batch_axis(posterior_states).detach().cpu()
        )
        forcing_batches.append(
            ensure_batch_axis(forcing).detach().cpu()
        )
        target_time_index_batches.append(
            rollout_target_time_indices(date, N_ROLLOUT_STEPS).unsqueeze(0)
        )

    dataset = RolloutTensorDataset(
        torch.cat(prior_batches, dim=0),
        torch.cat(posterior_batches, dim=0),
        torch.cat(forcing_batches, dim=0),
        torch.cat(target_time_index_batches, dim=0),
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        drop_last=drop_last,
    )


In [ ]:
# ### Un comment this once-off to save the training/validation information in /data
# fast_train_dl = make_fast_rollout_dl(
#     pipeline_i,
#     splits["train_split"],
#     batch_size=32,
#     shuffle=True,
#     desc="Building train rollout cache",
# )

# fast_valid_dl = make_fast_rollout_dl(
#     pipeline_i,
#     splits["valid_split"],
#     batch_size=32,
#     shuffle=False,
#     desc="Building valid rollout cache",
# )

# torch.save(
#     {
#         "prior": fast_train_dl.dataset.prior_states,
#         "posterior": fast_train_dl.dataset.posterior_states,
#         "forcing": fast_train_dl.dataset.forcing,
#         "target_time_indices": fast_train_dl.dataset.target_time_indices,
#     },
#     database +  "/OM2-emulator/data/" + "train_rollout_12step.pt",
# )

# torch.save(
#     {
#         "prior": fast_valid_dl.dataset.prior_states,
#         "posterior": fast_valid_dl.dataset.posterior_states,
#         "forcing": fast_valid_dl.dataset.forcing,
#         "target_time_indices": fast_valid_dl.dataset.target_time_indices,
#     },
#     database +  "/OM2-emulator/data/" + "valid_rollout_12step.pt",
# )

In [ ]:
train_cache = torch.load(
    database +  "/OM2-emulator/data/" + "train_rollout_12step.pt",
    map_location="cpu",
)

valid_cache = torch.load(
    database +  "/OM2-emulator/data/" + "valid_rollout_12step.pt",
    map_location="cpu",
)

# The cached forcing tensors are (N, n_steps, n_forcing_channels, H, W). If
# INCLUDE_WIND_STRESS_FORCING was toggled since these caches were last built,
# the channel count below will not match forcing_variables -- regenerate both
# caches from the (commented-out) cell above before continuing.
for name, cache in [("train_rollout_12step.pt", train_cache), ("valid_rollout_12step.pt", valid_cache)]:
    cached_channels = cache["forcing"].shape[2]
    if cached_channels != len(forcing_variables):
        raise ValueError(
            f"{name} has {cached_channels} cached forcing channel(s), but "
            f"forcing_variables currently defines {len(forcing_variables)} "
            f"({forcing_variables}). Regenerate the rollout caches (see the "
            "cell above) after changing INCLUDE_WIND_STRESS_FORCING."
        )

train_target_time_indices = train_cache.get(
    "target_time_indices",
    rollout_target_time_index_tensor(splits["train_split"], N_ROLLOUT_STEPS),
)
valid_target_time_indices = valid_cache.get(
    "target_time_indices",
    rollout_target_time_index_tensor(splits["valid_split"], N_ROLLOUT_STEPS),
)

train_dataset = RolloutTensorDataset(
    train_cache["prior"],
    train_cache["posterior"],
    train_cache["forcing"],
    train_target_time_indices,
)

valid_dataset = RolloutTensorDataset(
    valid_cache["prior"],
    valid_cache["posterior"],
    valid_cache["forcing"],
    valid_target_time_indices,
)

fast_train_dl = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

fast_valid_dl = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)


## Trainer

Unchanged from `AutoEncoder_om2.ipynb` -- same `L.Trainer` configuration and same
reason for bypassing PET's native `pyearthtools.training.lightning.Train` (too slow;
GitHub issue linked in the original notebook).

In [ ]:
print(fast_train_dl.dataset.posterior_states.shape)
print(fast_valid_dl.dataset.posterior_states.shape)

In [ ]:
def check_known_global_closure(dataloader, max_batches=None, warn_threshold=0.25):
    """Report cumulative closure mismatch on ground-truth rollout transitions."""
    losses = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            prior_states, posterior_states, forcing, target_time_indices = unpack_rollout_batch(batch)
            prior_states = ForwardLightningWrapper._squeeze_variable_axis(prior_states)
            posterior_states = ForwardLightningWrapper._squeeze_variable_axis(posterior_states)
            # forcing keeps its channel axis (B, n_steps, n_forcing_channels, H, W);
            # global_cumulative_closure_loss reads the heat-flux channel out of it.

            initial_ohc_norm = prior_states[:, 1:2]
            initial_time_index = target_time_indices[:, 0] - 1

            for step in range(N_ROLLOUT_STEPS):
                target_t = posterior_states[:, step : step + 1]
                target_idx = target_time_indices[:, step]

                known_loss = global_cumulative_closure_loss(
                    initial_ohc_norm=initial_ohc_norm,
                    predicted_ohc_norm=target_t,
                    forcing_sequence_norm=forcing[:, : step + 1],
                    initial_time_index=initial_time_index,
                    target_time_index=target_idx,
                    forcing_time_indices=target_time_indices[:, : step + 1],
                    area=lightning_model.area,
                    mask=lightning_model.mask,
                    ohc_mean=lightning_model.ohc_mean,
                    ohc_std=lightning_model.ohc_std,
                    forcing_mean=lightning_model.forcing_mean,
                    forcing_std=lightning_model.forcing_std,
                    dt_seconds=lightning_model.dt_seconds,
                    surface_flux_sign=lightning_model.surface_flux_sign,
                    closure_min_scale=lightning_model.closure_min_scale,
                    heat_flux_channel_index=lightning_model.heat_flux_channel_index,
                )
                losses.append(known_loss.detach().cpu())

    losses = torch.stack(losses)
    max_loss = losses.max().item()
    mean_loss = losses.mean().item()
    print(f"Known cumulative closure loss | mean={mean_loss:.6e}, max={max_loss:.6e}")

    if max_loss > warn_threshold:
        warnings.warn(
            "Known OHC/SF transitions do not exactly satisfy the cumulative closure "
            "diagnostic. Treat the closure term as a soft regulariser, not as an "
            "exact conservation law, unless a fully closed heat-budget tendency is used."
        )


if CLOSURE_WEIGHT > 0:
    check_known_global_closure(fast_train_dl)


In [ ]:
# PET's native training (pyearthtools.training.lightning.Train) is too slow
# for this setup -- see https://github.com/ACCESS-Community-Hub/PyEarthTools/issues/265

trainer = L.Trainer(
    max_epochs=200,
    num_sanity_val_steps=0,
    accelerator="gpu" if device.type == "cuda" else "cpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
)

trainer.fit(
    model=lightning_model,
    train_dataloaders=fast_train_dl,
    val_dataloaders=fast_valid_dl,
)

## Skill-test rollout: held-out real forcing

An autoregressive rollout against the held-out `skill_test_split` window, using real
surface flux forcing. This measures whether the deterministic ForwardUNet tracks
genuine forced OHC trends over the test period. The rollout starts from one
ground-truth initial condition, then feeds model predictions back into the next
step without injecting ground-truth OHC again.


In [ ]:
%%time
from tqdm.auto import tqdm 
# -----------------------------------------------------------------------------
# Cell purpose: autoregressive skill-test rollout against real held-out forcing.
# Runs the trained model forward from a single initial condition, supplying
# real flux at each step. Stores predictions and ground-truth for diagnostics.
# -----------------------------------------------------------------------------

base_model.eval()
base_model.to(device)

# Build one-step evaluation pipelines over the rollout window. The training
# pipeline uses N_ROLLOUT_STEPS=12 for recurrent loss, but this autoregressive
# loop only consumes the next month at each iteration. Keeping a separate
# one-step pipeline avoids loading and transforming 11 unused future months
# for every rollout step.
eval_state_window = petpipe.modifications.TemporalWindow(
    prior_indexes=[-1, 0],
    posterior_indexes=[1],
    timedelta=rollout_timedelta,
    merge_method=functools.partial(np.concatenate, axis=0),
)

eval_state_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *state_steps,
    eval_state_window,
)

eval_forcing_retrieval = petpipe.modifications.TemporalRetrieval(
    (1, 1),
    delta_unit="month",
    concat=True,
)

eval_forcing_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    *forcing_steps,
    eval_forcing_retrieval,
)

full_rollout_pipeline = petpipe.Pipeline(
    petpipe.branching.PipelineBranchPoint(
        eval_state_pipeline,
        eval_forcing_pipeline,
    ),
    iterator=full_rollout_split,
)

full_preds = []
full_targets = []

with torch.no_grad():
    prior_states = None
    full_rollout_dates = list(full_rollout_split)
    for date in tqdm(full_rollout_dates, desc="Full rollout"):        
        prior_window, target_window, forcing_window, _ = unpack_rollout_batch(full_rollout_pipeline[date])

        prior_window = ensure_batch_axis(prior_window).to(device=device, dtype=torch.float32)
        target_t = ensure_batch_axis(target_window).to(device=device, dtype=torch.float32)
        forcing_t = ensure_batch_axis(forcing_window).to(device=device, dtype=torch.float32)

        prior_window = ForwardLightningWrapper._squeeze_variable_axis(prior_window)
        target_t = ForwardLightningWrapper._squeeze_variable_axis(target_t)
        # forcing_t is (B, 1, n_forcing_channels, H, W) here -- eval_forcing_retrieval
        # only ever fetches a single timestep, so collapse that time axis (not the
        # channel axis, which _squeeze_variable_axis would target and which must be
        # kept when wind stress is stacked alongside heat flux).
        forcing_t = forcing_t[:, 0]

        # On the first step, seed from ground truth.
        if prior_states is None:
            prior_states = prior_window

        pred_t = base_model(prior_states, forcing_t, lightning_model.mask)

        full_preds.append(pred_t.cpu().numpy())
        full_targets.append(target_t.cpu().numpy())

        # Autoregressive: feed prediction back in, not ground truth.
        prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

full_preds = np.concatenate(full_preds, axis=0)
full_targets = np.concatenate(full_targets, axis=0)


## Control rollout: repeated flat forcing (isolates equilibrium drift)

An autoregressive rollout driven by the repeated, near-zero-net-flux control window
(`control_iterator`), rather than real forcing. By running the model under forcing
that carries minimal net heat input, long-term drift in OHC is interpreted as
compounding emulator bias rather than a response to a real forced trend.


In [ ]:
# %%time
# # -----------------------------------------------------------------------------
# # Cell purpose: long control rollout under repeated near-zero-net-flux forcing,
# # to isolate equilibrium drift from genuine forced trend response.
# # control_iterator chains the same 10-year DateRange n_control_repeats times
# # via SuperIterator -- see the splits section for the rationale.
# # -----------------------------------------------------------------------------

# # Build a control pipeline using the repeated flat-forcing iterator.
# control_pipeline = petpipe.Pipeline(
#     petpipe.branching.PipelineBranchPoint(
#         state_pipeline,
#         forcing_pipeline,
#     ),
#     iterator=control_iterator,
# )

# control_preds = []

# with torch.no_grad():
#     prior_states = None
#     full_control_dates = list(control_iterator)
#     for date in tqdm(full_control_dates, desc="Full control"):        
#         (prior_window, _), forcing_window = control_pipeline[date]
#         if prior_states is None:
#             prior_states = torch.tensor(
#                 prior_window, dtype=torch.float32, device=device
#             ).unsqueeze(0).squeeze(2)

#         forcing_t = torch.tensor(
#             forcing_window, dtype=torch.float32, device=device
#         ).unsqueeze(0).squeeze(2)

#         pred_t = base_model(prior_states, forcing_t, lightning_model.mask)

#         control_preds.append(pred_t.cpu().numpy())
#         prior_states = torch.cat([prior_states[:, 1:2], pred_t], dim=1)

# control_preds = np.concatenate(control_preds, axis=0)
# print(f'Control rollout complete. Array shape: {control_preds.shape}')

## Drift diagnostics

Two complementary diagnostics, matching Samudra's evaluation framework:

1. **Skill-test trend attenuation** under real held-out forcing. Compare the
   predicted global-mean OHC trend with the OM2 target trend.
2. **Control-rollout equilibrium drift** under repeated near-zero-net-flux forcing.
   A clean baseline should not create a large artificial century-scale trend when
   the repeated forcing cycle is close to balanced.

In [ ]:
%%time

# SKILL_TEST_TARGET_START/END are set in the time-configuration cell above --
# full_target_pipeline's targets exactly match full_rollout_split's targets
# (one month ahead of each date in full_rollout_split, i.e. this same window).
#
# mean/deviation are indexed here by POSITION via normalisation_time_lookup
# (the same lookup the training loss uses), not by mean.sel(time=slice(...)).
# A string time-slice can silently pick up one extra/fewer row than the exact
# calendar-month count PET's DateRange produces -- mean.time holds real,
# nearest-matched, mid-month timestamps, and slice-boundary matching against
# those isn't guaranteed to agree with a pure calendar-month count. Looking
# up each target date's exact position instead guarantees full_mean/
# full_deviation line up 1:1 with full_preds/full_targets by construction.
skill_test_target_dates = list(
    petpipe.iterators.DateRange(SKILL_TEST_TARGET_START, SKILL_TEST_TARGET_END, interval=ROLLOUT_INTERVAL)
)
skill_test_target_indices = [
    normalisation_time_lookup[pd.Timestamp(str(d)).strftime("%Y-%m")]
    for d in skill_test_target_dates
]
full_mean = mean.isel(time=skill_test_target_indices)
full_deviation = deviation.isel(time=skill_test_target_indices)

assert len(skill_test_target_dates) == full_preds.shape[0], (
    f"full_target date count ({len(skill_test_target_dates)}) does not match "
    f"full_preds ({full_preds.shape[0]}) -- re-run the full-rollout cell above "
    "so full_preds/full_targets and the time-configuration cell agree."
)

full_normalisation = petpipe.operations.xarray.normalisation.Evaluated(
    normalisation_eval="(sample - mean) / deviation",
    unnormalisation_eval="(sample * deviation) + mean",
    mean=full_mean,
    deviation=full_deviation,
)

full_target_pipeline = petpipe.Pipeline(
    ACCESS_OHC_accessor,
    _shared_coord_drop,
    _shared_var_drop,
    petpipe.operations.xarray.select.SelectDataset(["ocean_heat_content_2d"]),
    petpipe.operations.xarray.Sort(order=["ocean_heat_content_2d"], strict=True),
    petpipe.operations.xarray.reshape.Dimensions(["time", "latitude", "longitude"]),
    full_normalisation,
    petdata.transforms.coordinates.Drop(['geolat_t', 'geolon_t', 'xu_ocean', 'yu_ocean'], ignore_missing=True),
    petpipe.operations.xarray.values.FillNan(0, posinf=0, neginf=0),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'),
    iterator=petpipe.iterators.DateRange(SKILL_TEST_TARGET_START, SKILL_TEST_TARGET_END, interval=ROLLOUT_INTERVAL),
)

_ = full_target_pipeline[SKILL_TEST_TARGET_START]

full_predicted_data = full_target_pipeline.undo(full_preds)
full_target_data = full_target_pipeline.undo(full_targets)

## Plots

This section provides three diagnostics for the rollout: spatial OHC anomaly maps, a global-integrated OHC anomaly time series, and a global RMSE time series.

- Spatial maps accept either months (`YYYY-MM`) or full years (`YYYY`). Year inputs are averaged over all available months in that year.
- The anomaly reference is the training-period monthly climatology derived from data through `2015-03`.

In [ ]:
# -----------------------------------------------------------------------------
# Cell purpose: prepare the rollout diagnostics used by the plotting cells.
# -----------------------------------------------------------------------------

def rename_om2_grid(data_array):
    rename_map = {}
    if "yt_ocean" in data_array.dims:
        rename_map["yt_ocean"] = "latitude"
    if "xt_ocean" in data_array.dims:
        rename_map["xt_ocean"] = "longitude"
    return data_array.rename(rename_map) if rename_map else data_array


def select_month(data_array, month):
    month_period = pd.Period(pd.Timestamp(month), freq="M")
    available_months = pd.PeriodIndex(
        pd.to_datetime(data_array.time.values),
        freq="M",
    )
    matches = np.flatnonzero(available_months == month_period)
    if len(matches) == 0:
        raise KeyError(
            f"Month {month_period} not found. Available range is "
            f"{available_months.min()} to {available_months.max()}."
        )
    return data_array.isel(time=int(matches[0]))


def global_integrated_ohc(ohc_field):
    return (ohc_field * area).sum(["latitude", "longitude"], skipna=True)


def global_rmse_timeseries(prediction, truth):
    squared_error = (prediction - truth) ** 2
    return np.sqrt(
        (squared_error * area).sum(["latitude", "longitude"], skipna=True)
        / area.sum(["latitude", "longitude"])
    )


# time_start is used only to fetch a valid date; area_t is time-invariant.
area = rename_om2_grid(ACCESS_OHC_accessor[time_start]["area_t"])
ocean_mask = rename_om2_grid(mask).fillna(0)
area = area.where(ocean_mask == 1).fillna(0).isel(time=0,drop=True)

predicted_ohc = rename_om2_grid(full_predicted_data.ocean_heat_content_2d)
target_ohc = rename_om2_grid(full_target_data.ocean_heat_content_2d)
ohc_normalisation_mean = rename_om2_grid(mean["ocean_heat_content_2d"])

rollout_time = pd.DatetimeIndex(pd.to_datetime(target_ohc.time.values))
verification_dates = rollout_time.to_period("M").to_timestamp()
ohc_units = target_ohc.attrs.get("units", "native OHC units")

# Keep the climatology reference on the original rollout timestamps so xarray
# subtraction aligns month-by-month without dropping the time dimension.
reference_ohc = xr.concat(
    [select_month(ohc_normalisation_mean, month) for month in rollout_time],
    dim=pd.Index(rollout_time, name="time"),
)
predicted_ohc_anomaly = predicted_ohc - reference_ohc
target_ohc_anomaly = target_ohc - reference_ohc

monthly_global_ohc_pred = global_integrated_ohc(predicted_ohc_anomaly)
monthly_global_ohc_truth = global_integrated_ohc(target_ohc_anomaly)
monthly_global_rmse = global_rmse_timeseries(predicted_ohc, target_ohc)

In [ ]:
# Enter months as "YYYY-MM" or years as "YYYY". Each entry is plotted on its own row.
SNAPSHOT_PERIODS = ["2005-05","2015-12","2010-12"]
ANOMALY_SCALE = 2e9#"auto"
DIFFERENCE_SCALE = 2e9#"auto"

predicted_annual_anomaly = predicted_ohc_anomaly.groupby("time.year").mean("time")
target_annual_anomaly = target_ohc_anomaly.groupby("time.year").mean("time")
annual_difference = (predicted_ohc - target_ohc).groupby("time.year").mean("time")


def select_snapshot_fields(period_label):
    period_text = str(period_label)
    if len(period_text) == 4 and period_text.isdigit():
        year = int(period_text)
        available_years = predicted_annual_anomaly.year.values.tolist()
        if year not in available_years:
            raise KeyError(
                f"Year {year} not found. Available years are {available_years}."
            )
        return (
            predicted_annual_anomaly.sel(year=year),
            target_annual_anomaly.sel(year=year),
            annual_difference.sel(year=year),
            f"{year} mean",
        )

    timestamp = pd.Timestamp(period_text)
    return (
        select_month(predicted_ohc_anomaly, timestamp),
        select_month(target_ohc_anomaly, timestamp),
        select_month(predicted_ohc, timestamp) - select_month(target_ohc, timestamp),
        timestamp.strftime("%Y-%m"),
    )


snapshot_fields = [select_snapshot_fields(period) for period in SNAPSHOT_PERIODS]

if ANOMALY_SCALE == "auto":
    anomaly_limit = max(
        [1.0]
        + [
            float(np.nanmax(np.abs(field.values)))
            for predicted_anomaly, target_anomaly, _, _ in snapshot_fields
            for field in (predicted_anomaly, target_anomaly)
            if np.isfinite(field.values).any()
        ]
    )
else:
    anomaly_limit = float(ANOMALY_SCALE)

if DIFFERENCE_SCALE == "auto":
    difference_limit = max(
        [1.0]
        + [
            float(np.nanmax(np.abs(difference.values)))
            for _, _, difference, _ in snapshot_fields
            if np.isfinite(difference.values).any()
        ]
    )
else:
    difference_limit = float(DIFFERENCE_SCALE)

fig, axes = plt.subplots(
    len(SNAPSHOT_PERIODS),
    3,
    figsize=(15, 4 * len(SNAPSHOT_PERIODS)),
    squeeze=False,
    constrained_layout=True,
)

for row_index, (predicted_anomaly, target_anomaly, difference, label) in enumerate(
    snapshot_fields
):
    predicted_plot = predicted_anomaly.plot(
        ax=axes[row_index, 0],
        cmap="RdBu_r",
        vmin=-anomaly_limit,
        vmax=anomaly_limit,
        add_colorbar=False,
    )
    target_anomaly.plot(
        ax=axes[row_index, 1],
        cmap="RdBu_r",
        vmin=-anomaly_limit,
        vmax=anomaly_limit,
        add_colorbar=False,
    )
    difference_plot = difference.plot(
        ax=axes[row_index, 2],
        cmap="RdBu_r",
        vmin=-difference_limit,
        vmax=difference_limit,
        add_colorbar=False,
    )

    axes[row_index, 0].set_title(f"Predicted anomaly | {label}")
    axes[row_index, 1].set_title(f"Target anomaly | {label}")
    axes[row_index, 2].set_title(f"Prediction minus truth | {label}")

    for ax in axes[row_index]:
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    fig.colorbar(
        predicted_plot,
        ax=axes[row_index, :2],
        shrink=0.92,
        pad=0.02,
        label=f"OHC anomaly ({ohc_units})",
    )
    fig.colorbar(
        difference_plot,
        ax=axes[row_index, 2],
        shrink=0.92,
        pad=0.02,
        label=f"Prediction minus truth ({ohc_units})",
    )

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)

ax.plot(
    verification_dates,
    monthly_global_ohc_pred.values / 1e22,
    label="Predicted",
    linewidth=2.0,
)
ax.plot(
    verification_dates,
    monthly_global_ohc_truth.values / 1e22,
    label="Truth",
    linewidth=2.0,
)
ax.axvline(
    verification_dates[0],
    color="0.4",
    linestyle="--",
    linewidth=1.0,
)

ax.set_title("Global-integrated OHC anomaly through the rollout")
ax.set_xlabel("Verification month")
ax.set_ylabel("Integrated OHC anomaly ($10^{22}$ J)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False, loc="best")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)

ax.plot(
    verification_dates,
    monthly_global_rmse.values,
    color="firebrick",
    linewidth=2.0,
)
ax.axvline(
    verification_dates[0],
    color="0.4",
    linestyle="--",
    linewidth=1.0,
)

ax.set_title("Global area-weighted RMSE of the OHC field")
ax.set_xlabel("Verification month")
ax.set_ylabel(f"RMSE ({ohc_units})")
ax.grid(True, alpha=0.3)
plt.show()

## Closing note

This notebook now runs one clean deterministic baseline: `ForwardUNet` with
multi-step recurrent masked MSE. The skill-test and control-rollout diagnostics
are intended to characterise that baseline before adding any extra modelling
complexity elsewhere.
